# Circular Poisson spatial scan lab
Find elevated-rate circular clusters in areal case counts while adjusting for population at risk, then calibrate the maximum scan statistic by conditional Monte Carlo simulation.


In [ ]:
from pathlib import Path
import sys
for parent in (Path.cwd(), *Path.cwd().parents):
    package = parent / "implementations" / "python" / "point-process"
    if package.is_dir():
        sys.path.insert(0, str(package))
        break
else:
    raise RuntimeError("Could not locate implementations/python/point-process")
import numpy as np
from spatial_scan import poisson_spatial_scan, monte_carlo_poisson_scan


## 1. Create a synthetic areal dataset
Each row is an areal unit represented by a projected planar centroid, a case count and a population-at-risk value.


In [ ]:
coordinates = np.array([[0., 0.], [1., 0.], [10., 0.], [11., 0.], [20., 0.]])
cases = np.array([9, 8, 1, 1, 1])
population = np.full(5, 100.)
print("Total cases:", cases.sum(), "Total population:", population.sum())


## 2. Find the most likely elevated cluster
Candidate circles are centred at observed centroids and expand through distinct centroid distances. Tied distances are included together.


In [ ]:
observed = poisson_spatial_scan(coordinates, cases, population,
                                 max_population_fraction=0.5, min_cases=2)
observed


## 3. Calibrate the maximum statistic
The simulation conditions on the observed total case count and allocates cases according to population shares. Increase the replication count for finer p-value resolution.


In [ ]:
calibrated = monte_carlo_poisson_scan(coordinates, cases, population,
                                      n_simulations=999, rng=42)
print("Observed maximum LLR:", calibrated["observed"]["log_likelihood_ratio"])
print("Monte Carlo p-value:", calibrated["p_value"])
print("Replications:", calibrated["n_simulations"])


## 4. Sensitivity to the maximum population fraction
The candidate-zone cap changes the set of circles considered. Compare the selected zone as the maximum allowed population share changes.


In [ ]:
for cap in (0.2, 0.3, 0.5, 0.8):
    result = poisson_spatial_scan(coordinates, cases, population,
                                  max_population_fraction=cap, min_cases=2)
    print(cap, result["member_indices"], result["log_likelihood_ratio"])


## Interpretation and limits
This is a bounded centroid-based circular Poisson scan, not a full SaTScan reproduction. It does not implement covariates, temporal scanning, elliptical or arbitrary zones, secondary clusters or polygon-boundary geometry. Use projected coordinates with meaningful linear units, justify the candidate-zone family and choose enough Monte Carlo replications for the required p-value resolution.
